In [ ]:
!pip install shap

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import shap

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
fraud = pd.read_csv(
    "../data/processed/fraud_processed.csv"
)

fraud.head()

In [ ]:
X = fraud.drop(
    columns=["class"]
)

y = fraud["class"]

In [ ]:
bool_cols = X.select_dtypes(
    include=["bool"]
).columns

for col in bool_cols:
    X[col] = X[col].astype(int)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
best_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

best_model.fit(
    X_train,
    y_train
)

In [ ]:
feature_importance = pd.DataFrame(
    {
        "Feature": X.columns,
        "Importance": best_model.feature_importances_
    }
)

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance.head(10)

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    data=feature_importance.head(10),
    x="Importance",
    y="Feature"
)

plt.title(
    "Top 10 Random Forest Features"
)

plt.tight_layout()

plt.show()

In [ ]:
explainer = shap.TreeExplainer(
    best_model
)

shap_values = explainer.shap_values(
    X_test
)

In [ ]:
shap.summary_plot(
    shap_values[1],
    X_test
)

In [ ]:
y_pred = best_model.predict(
    X_test
)

results = X_test.copy()

results["actual"] = y_test.values

results["predicted"] = y_pred

In [ ]:
tp_index = results[
    (results["actual"] == 1)
    &
    (results["predicted"] == 1)
].index[0]

tp_index

In [ ]:
fp_index = results[
    (results["actual"] == 0)
    &
    (results["predicted"] == 1)
].index[0]

fp_index

In [ ]:
fn_index = results[
    (results["actual"] == 1)
    &
    (results["predicted"] == 0)
].index[0]

fn_index

In [ ]:
shap.force_plot(
    explainer.expected_value[1],
    shap_values[1][
        X_test.index.get_loc(tp_index)
    ],
    X_test.loc[tp_index],
    matplotlib=True
)

In [ ]:
shap.force_plot(
    explainer.expected_value[1],
    shap_values[1][
        X_test.index.get_loc(fp_index)
    ],
    X_test.loc[fp_index],
    matplotlib=True
)

In [ ]:
shap.force_plot(
    explainer.expected_value[1],
    shap_values[1][
        X_test.index.get_loc(fn_index)
    ],
    X_test.loc[fn_index],
    matplotlib=True
)

In [ ]:
feature_importance.head(5)

In [ ]:
# Business Recommendations

1. Transactions occurring shortly after account creation should trigger additional verification.

2. Users exhibiting unusually high transaction frequency should receive increased fraud monitoring.

3. High-risk geographic regions identified through IP-country mapping should receive enhanced authentication checks.

4. Transactions occurring during unusual hours should be flagged for review.

5. Device reuse across multiple accounts should be monitored as a potential fraud indicator.